In [1]:
from collections import deque

# --- 1. CLASES BASE ---
class Problem:
    def __init__(self, initial, goal=None):
        self.initial = initial
        self.goal = goal
    
    def actions(self, state):
        raise NotImplementedError
    
    def result(self, state, action):
        raise NotImplementedError
    
    def goal_test(self, state):
        return state == self.goal
    
    def path_cost(self, c, state1, action, state2):
        return c + 1
    
    def h(self, state):
        """Función heurística por defecto"""
        return 0

class MetroProblem(Problem):
    def __init__(self, initial, goal, graph, heuristicas=None):
        super().__init__(initial, goal)
        self.graph = graph
        self.heuristicas = heuristicas or {}
        
    def actions(self, state):
        return list(self.graph.get(state, {}).keys())
    
    def result(self, state, action):
        return action
        
    def h(self, state):
        # Devuelve el valor heurístico hacia la meta; si no está definido, regresa 0
        return self.heuristicas.get(state, 0)

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        
    def expand(self, problem):
        return [self.child_node(problem, action) for action in problem.actions(self.state)]
    
    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        new_cost = problem.path_cost(self.path_cost, self.state, action, next_state)
        return Node(state=next_state, parent=self, action=action, path_cost=new_cost)
    
    def solution(self):
        node = self
        path = []
        while node.parent:
            path.append(node.state)
            node = node.parent
        path.append(node.state)
        return path[::-1]


# --- 2. ALGORITMO HILL CLIMBING ---
def hill_climbing(problem):
    """
    Algoritmo de Hill Climbing (Ascensión de Colinas).
    Se mueve al vecino con menor valor heurístico (más cercano a la meta).
    """
    current = Node(problem.initial)
    
    while True:
        # Si ya llegamos a la meta, regresamos el nodo actual
        if problem.goal_test(current.state):
            return current
            
        neighbors = current.expand(problem)
        if not neighbors:
            break
            
        # Seleccionar el vecino con el menor valor de h(n) (minimizar distancia a la meta)
        best_neighbor = min(neighbors, key=lambda node: problem.h(node.state))
        
        # Si el vecino no es mejor que el estado actual, nos detenemos (máximo local / meseta)
        if problem.h(best_neighbor.state) >= problem.h(current.state):
            return current
            
        current = best_neighbor


# --- 3. GRAFO DEL METRO DE LA CDMX ---
grafo_metro = {
    'Cuatro Caminos': {'Panteones': 1},
    'Panteones': {'Cuatro Caminos': 1, 'Tacuba': 1},
    'Tacuba': {'Panteones': 1, 'Cuitláhuac': 1, 'San Joaquín': 1},
    'Cuitláhuac': {'Tacuba': 1, 'Popotla': 1},
    'Popotla': {'Cuitláhuac': 1, 'Colegio Militar': 1},
    'Colegio Militar': {'Popotla': 1, 'Normal': 1},
    'Normal': {'Colegio Militar': 1, 'San Cosme': 1},
    'San Cosme': {'Normal': 1, 'Revolución': 1},
    'Revolución': {'San Cosme': 1, 'Hidalgo': 1},
    'Hidalgo': {'Revolución': 1, 'Bellas Artes': 1, 'Juárez': 1, 'Guerrero': 1},
    'Bellas Artes': {'Hidalgo': 1, 'Allende': 1, 'San Juan de Letrán': 1, 'Garibaldi': 1},
    'Juárez': {'Hidalgo': 1, 'Balderas': 1},
    'Balderas': {'Juárez': 1, 'Niños Héroes': 1, 'Salto del Agua': 1, 'Cuauhtémoc': 1},
    'Niños Héroes': {'Balderas': 1, 'Hospital General': 1},
    'Hospital General': {'Niños Héroes': 1, 'Centro Médico': 1},
    'Centro Médico': {'Hospital General': 1, 'Etiopía': 1, 'Chabacano': 1, 'Insurgentes': 1},
    'Etiopía': {'Centro Médico': 1, 'Eugenia': 1},
    'Eugenia': {'Etiopía': 1, 'División del Norte': 1},
    'División del Norte': {'Eugenia': 1, 'Zapata': 1},
    
    'Zapata': {'División del Norte': 1, 'Coyoacán': 1, 'Parque de los Venados': 1, 'Ermita': 1},
    
    'Chabacano': {'Centro Médico': 1, 'San Antonio Abad': 1, 'La Viga': 1, 'Jamaica': 1, 'Viaducto': 1},
    'Viaducto': {'Chabacano': 1, 'Xola': 1},
    'Xola': {'Viaducto': 1, 'Villa de Cortés': 1},
    'Villa de Cortés': {'Xola': 1, 'Nativitas': 1},
    'Nativitas': {'Villa de Cortés': 1, 'Portales': 1},
    'Portales': {'Nativitas': 1, 'Ermita': 1},
    'Ermita': {'Portales': 1, 'Taxqueña': 1, 'Zapata': 1},
    'Taxqueña': {'Ermita': 1},

    'Jamaica': {'Chabacano': 1, 'Mixiuhca': 1, 'Candelaria': 1, 'Fray Servando': 1},
    'Mixiuhca': {'Jamaica': 1, 'Velodromo': 1},
    'Velodromo': {'Mixiuhca': 1, 'Ciudad Deportiva': 1},
    'Ciudad Deportiva': {'Velodromo': 1, 'Pantitlán': 1},
    'Pantitlán': {'Ciudad Deportiva': 1, 'Zaragoza': 1, 'Agrícola Oriental': 1, 'Oceanía': 1, 'San Lázaro': 1},

    'Politécnico': {'Instituto del Petróleo': 1},
    'Instituto del Petróleo': {'Politécnico': 1, 'Autobuses del Norte': 1, 'Deportivo 18 de Marzo': 1},
    'Autobuses del Norte': {'Instituto del Petróleo': 1, 'La Raza': 1},
    'La Raza': {'Autobuses del Norte': 1, 'Tlatelolco': 1, 'Mysterios': 1},
    'Tlatelolco': {'La Raza': 1, 'Guerrero': 1},
    'Guerrero': {'Tlatelolco': 1, 'Hidalgo': 1, 'Garibaldi': 1},
    'Garibaldi': {'Guerrero': 1, 'Bellas Artes': 1},
    'San Lázaro': {'Pantitlán': 1, 'Morelos': 1},
    'Morelos': {'San Lázaro': 1, 'Candelaria': 1},
    'Oceanía': {'Pantitlán': 1, 'Terminal Aérea': 1, 'Aragón': 1}
}


# --- 4. HEURÍSTICAS Y EJECUCIÓN DE RUTAS ---
# Definimos heurísticas sencillas (ej. estimación de saltos restantes orientados a cada meta)
h_pantitlan = {'Pantitlán': 0, 'Ciudad Deportiva': 1, 'Velodromo': 2, 'Mixiuhca': 3, 'Jamaica': 4, 'Chabacano': 5}
h_taxquena = {'Taxqueña': 0, 'Ermita': 1, 'Portales': 2, 'Nativitas': 3, 'Villa de Cortés': 4, 'Xola': 5, 'Viaducto': 6, 'Chabacano': 7}
h_oceania = {'Oceanía': 0, 'Pantitlán': 1, 'San Lázaro': 2, 'Morelos': 3, 'Candelaria': 4}

rutas_solicitadas = [
    ("Cuatro Caminos", "Pantitlán", h_pantitlan),
    ("Politécnico", "Taxqueña", h_taxquena),
    ("Zapata", "Oceanía", h_oceania)
]

for origen, destino, heuristica in rutas_solicitadas:
    print(f"\n" + "="*50)
    print(f"Ruta (Hill Climbing): {origen} -> {destino}")
    print("="*50)
    
    problema = MetroProblem(origen, destino, grafo_metro, heuristica)
    resultado_hc = hill_climbing(problema)
    
    if resultado_hc and resultado_hc.state == destino:
        print(f"[Hill Climbing] ¡Meta alcanzada! ({resultado_hc.path_cost} estaciones):")
        print(" -> ".join(resultado_hc.solution()))
    else:
        print(f"[Hill Climbing] Se quedó en un máximo local en la estación: '{resultado_hc.state}'")
        print("Camino recorrido hasta ahí:", " -> ".join(resultado_hc.solution()))


Ruta (Hill Climbing): Cuatro Caminos -> Pantitlán
[Hill Climbing] Se quedó en un máximo local en la estación: 'Cuatro Caminos'
Camino recorrido hasta ahí: Cuatro Caminos

Ruta (Hill Climbing): Politécnico -> Taxqueña
[Hill Climbing] Se quedó en un máximo local en la estación: 'Politécnico'
Camino recorrido hasta ahí: Politécnico

Ruta (Hill Climbing): Zapata -> Oceanía
[Hill Climbing] Se quedó en un máximo local en la estación: 'Zapata'
Camino recorrido hasta ahí: Zapata
